# Visual Storyteller — Data Loading & Model Training

**Final Project | Image Captioning with CNN + Transformer**

---

The core task here is deceptively simple to state: given an image, produce a sentence that describes it. In practice, this sits at the intersection of computer vision and natural language generation — two domains that require very different representations of the world. Images are dense grids of pixel values; sentences are discrete sequences of tokens with syntactic and semantic structure. Bridging these two modalities is what makes image captioning genuinely interesting.

In this notebook I document my full pipeline:
1. **Data loading & exploration** — understanding what the dataset looks like before touching a model
2. **Preprocessing** — building a vocabulary, tokenising captions, and setting up `Dataset`/`DataLoader` objects
3. **Model definition** — an EfficientNet encoder paired with a Transformer decoder
4. **Training loop** — with gradient clipping, learning-rate scheduling, and checkpoint saving
5. **Loss curves** — so we can visually confirm the model is actually learning

Everything is written to be reproducible: I set seeds at the top and save all artefacts to disk so the inference notebook can pick up from where this one finishes.

## 0. Imports & Configuration

I keep all hyper-parameters in a single `Config` dataclass at the top. This makes it trivial to run ablations — change one value here and re-run, rather than hunting through the notebook for magic numbers.

In [ ]:
import sys
print(sys.executable)


import os
import re
import json
import random
import pickle
import logging
import math
import time
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as transforms

# timm is required — no silent fallback, because a fallback encoder would
# create a cfg/weights mismatch that breaks inference.ipynb.
try:
    import timm
except ImportError:
    raise ImportError(
        "This project requires timm. Install it with:\n"
        "    pip install timm"
    )

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)


# ── Reproducibility ────────────────────────────────────────────────────────────
# Cover every major source of randomness. Note: some CUDA ops remain
# non-deterministic by design; warn_only=True lets training continue
# while still flagging genuinely non-deterministic operations.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)  # dict/set ordering in CPython

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)   # multi-GPU

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False  # disables auto-tuning (non-deterministic)

torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """
    Seed each DataLoader worker so that augmentation ops (random crop,
    colour jitter, etc.) are reproducible. We derive the seed from
    torch.initial_seed(), which PyTorch already sets per-worker based on
    the generator — this is more robust than SEED + worker_id because it
    accounts for the epoch-level re-seeding PyTorch does internally.
    """
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Shared generator passed to DataLoaders that shuffle data
_g = torch.Generator()
_g.manual_seed(SEED)


@dataclass
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────────
    data_root: str = "./caption_data"          # unzipped dataset folder
    images_dir: str = "./caption_data/images"  # folder containing .jpg files
    captions_file: str = "./caption_data/captions.txt"  # TSV/CSV with image,caption cols
    output_dir: str = "./artifacts"            # checkpoints + vocab saved here

    # ── Vocabulary ─────────────────────────────────────────────────────────────
    vocab_min_freq: int = 2        # tokens appearing fewer times are replaced with <UNK>
    max_caption_len: int = 52      # captions longer than this are truncated

    # ── Model ──────────────────────────────────────────────────────────────────
    encoder_name: str = "efficientnet_b3"  # any timm model with features_only=True
    encoder_pretrained: bool = True
    embed_dim: int = 512
    num_heads: int = 8
    num_decoder_layers: int = 4
    feedforward_dim: int = 2048
    dropout: float = 0.1
    fine_tune_encoder: bool = True   # unfreeze encoder after warmup

    # ── Training ───────────────────────────────────────────────────────────────
    batch_size: int = 32
    num_epochs: int = 30
    warmup_epochs: int = 3          # encoder stays frozen for this many epochs
    learning_rate: float = 3e-4
    encoder_lr_multiplier: float = 0.1  # encoder LR = lr * this
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.1
    val_split: float = 0.1
    test_split: float = 0.1   # held out entirely — only touched in inference.ipynb
    num_workers: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # ── Image ──────────────────────────────────────────────────────────────────
    image_size: int = 224


cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

logger.info(f"Device: {cfg.device}")
logger.info(f"Output directory: {cfg.output_dir}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading & Exploration

Before I build anything, I want to understand the data. The dataset pairs each of the 8,000 images with five human-written captions — that's 40,000 caption strings in total. It's worth spending a few minutes looking at caption length distributions and vocabulary size, because these directly influence model hyper-parameters like `max_caption_len` and `embed_dim`.

The captions file can be in various formats depending on the dataset version (Flickr8k uses a `image_name\tcaption` TSV). I handle the most common layouts below.

In [ ]:
def load_captions(captions_file: str) -> Dict[str, List[str]]:
    """
    Load captions from file into a dict mapping image filename -> list of captions.
    Handles TSV (image\tcaption) and CSV (image,caption) formats automatically.
    Skips header rows and filters out entries whose image file does not exist.
    """
    # Known column-header values that should be skipped
    HEADER_TOKENS = {"image", "image_name", "filename", "file_name", "img", "img_id"}

    captions_map: Dict[str, List[str]] = {}
    path = Path(captions_file)

    if not path.exists():
        for alt in ["captions.txt", "descriptions.txt", "Flickr8k.token.txt",
                    "captions.csv", "annotations.txt"]:
            candidate = path.parent / alt
            if candidate.exists():
                path = candidate
                logger.info(f"Found captions at: {path}")
                break
        else:
            raise FileNotFoundError(
                f"Could not find a captions file in {path.parent}. "
                "Please set cfg.captions_file to the correct path."
            )

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            # Detect delimiter
            sep = "\t" if "\t" in line else ","
            parts = line.split(sep, 1)
            if len(parts) != 2:
                continue

            img_id, caption = parts[0].strip(), parts[1].strip()

            # Skip header rows (e.g. CSV files that start with "image,caption")
            if img_id.lower() in HEADER_TOKENS:
                continue

            # Flickr8k uses "image.jpg#0" notation — strip the caption index
            img_name = img_id.split("#")[0]

            captions_map.setdefault(img_name, []).append(caption)

    raw_count = len(captions_map)

    # Remove entries whose image file is not present on disk.
    # This prevents silent KeyErrors later and makes the dataset count accurate.
    captions_map = {
        img: caps
        for img, caps in captions_map.items()
        if (Path(cfg.images_dir) / img).exists()
    }

    missing = raw_count - len(captions_map)
    if missing:
        logger.warning(f"{missing} caption entries had no matching image on disk and were removed.")
    logger.info(f"Loaded captions for {len(captions_map):,} images.")
    return captions_map


captions_map = load_captions(cfg.captions_file)

# Quick sanity check
sample_key = next(iter(captions_map))
print(f"\nSample image: {sample_key}")
print("Captions:")
for c in captions_map[sample_key]:
    print(f"  • {c}")

In [ ]:
# ── Exploratory analysis ───────────────────────────────────────────────────────
all_captions = [cap for caps in captions_map.values() for cap in caps]
caption_lengths = [len(c.split()) for c in all_captions]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Caption length histogram
axes[0].hist(caption_lengths, bins=40, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].axvline(np.percentile(caption_lengths, 95), color="tomato", linestyle="--",
                label=f"95th pct = {np.percentile(caption_lengths, 95):.0f} tokens")
axes[0].set_xlabel("Caption length (words)")
axes[0].set_ylabel("Count")
axes[0].set_title("Caption Length Distribution")
axes[0].legend()

# Vocabulary frequency (top 30 words)
all_words = [w.lower() for cap in all_captions for w in cap.split()]
word_freq = Counter(all_words)
top_words = word_freq.most_common(30)
words, freqs = zip(*top_words)
axes[1].barh(list(reversed(words)), list(reversed(freqs)), color="darkorange")
axes[1].set_xlabel("Frequency")
axes[1].set_title("Top 30 Most Common Words")

plt.tight_layout()
plt.savefig(f"{cfg.output_dir}/eda_distributions.png", dpi=120)
plt.show()

print(f"\nDataset statistics:")
print(f"  Total images:       {len(captions_map):,}")
print(f"  Total captions:     {len(all_captions):,}")
print(f"  Avg caption length: {np.mean(caption_lengths):.1f} words")
print(f"  Max caption length: {max(caption_lengths)} words")
print(f"  Unique tokens:      {len(word_freq):,}")

In [ ]:
# ── Visualise a few image–caption pairs ────────────────────────────────────────
sample_keys = random.sample(list(captions_map.keys()), 4)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, img_name in zip(axes, sample_keys):
    img_path = Path(cfg.images_dir) / img_name
    if img_path.exists():
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
    ax.set_title("\n".join([c[:45] + "..." if len(c) > 45 else c
                             for c in captions_map[img_name][:2]]),
                 fontsize=7, loc="left")
    ax.axis("off")

plt.suptitle("Sample Images with Ground-Truth Captions", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{cfg.output_dir}/eda_samples.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Train / Validation / Test Split

I create a **three-way split** before touching the vocabulary, for two reasons:

- **No leakage into the vocab** — the vocabulary is built from training captions only. Val and test captions play no part in frequency counting.
- **Clean evaluation roles** — the val set is used *only* to pick the best checkpoint during training. The test set is never looked at until the inference notebook runs the final demo and BLEU evaluation. Using the val set for both checkpoint selection and final reporting would give an optimistic score because hyper-parameter choices were implicitly guided by val performance.

The split is seeded through an explicit `random.Random(SEED)` instance rather than the global `random` state, so it stays stable even if other cells are re-run in a different order.

Split sizes: 80% train / 10% val / 10% test — saved to `artifacts/split.json`.

In [ ]:
# Use an explicit seeded RNG instance — not the global random state — so this
# cell is stable regardless of execution order.
rng = random.Random(SEED)
all_images = list(captions_map.keys())
rng.shuffle(all_images)

n = len(all_images)
n_test = int(n * cfg.test_split)
n_val  = int(n * cfg.val_split)

test_images  = set(all_images[:n_test])
val_images   = set(all_images[n_test: n_test + n_val])
train_images = set(all_images[n_test + n_val:])

train_captions = {k: v for k, v in captions_map.items() if k in train_images}
val_captions   = {k: v for k, v in captions_map.items() if k in val_images}
test_captions  = {k: v for k, v in captions_map.items() if k in test_images}

# Persist the split — inference.ipynb uses test_images for demo + BLEU
split_path = f"{cfg.output_dir}/split.json"
with open(split_path, "w") as f:
    json.dump(
        {
            "train_images": sorted(list(train_images)),
            "val_images":   sorted(list(val_images)),
            "test_images":  sorted(list(test_images)),
        },
        f, indent=2,
    )

logger.info(
    f"Split saved → {split_path}  "
    f"(train={len(train_images)}, val={len(val_images)}, test={len(test_images)})"
)
print(f"Train: {len(train_images):,} images  |  Val: {len(val_images):,}  |  Test: {len(test_images):,}")
assert not (train_images & val_images), "Overlap between train and val!"
assert not (train_images & test_images), "Overlap between train and test!"
assert not (val_images & test_images), "Overlap between val and test!"
print("✓ No overlap between splits.")

## 3. Vocabulary & Text Preprocessing

I build a character-free, word-level vocabulary from **training captions only** — the validation split plays no part here. Words that appear fewer than `vocab_min_freq` times get mapped to `<UNK>`.

Special tokens:
- `<PAD>` (idx 0) — pads sequences to equal length in a batch
- `<SOS>` (idx 1) — start-of-sequence, prepended to every target during teacher forcing
- `<EOS>` (idx 2) — end-of-sequence, the model learns to emit this when it finishes
- `<UNK>` (idx 3) — unknown / out-of-vocabulary token

In [ ]:
class Vocabulary:
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"

    def __init__(self, min_freq: int = 2):
        self.min_freq = min_freq
        self.itos: List[str] = []       # index -> token
        self.stoi: Dict[str, int] = {}  # token -> index

    def build(self, captions: List[str]) -> None:
        """Build vocab from a flat list of caption strings."""
        freq = Counter()
        for cap in captions:
            freq.update(self._tokenise(cap))
        specials = [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN, self.UNK_TOKEN]
        self.itos = specials + sorted(
            [w for w, f in freq.items() if f >= self.min_freq]
        )
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}
        logger.info(
            f"Vocabulary size: {len(self.itos):,} tokens "
            f"(min_freq={self.min_freq}, "
            f"pruned {sum(1 for f in freq.values() if f < self.min_freq):,} rare words)"
        )

    @staticmethod
    def _tokenise(text: str) -> List[str]:
        text = text.lower().strip()
        text = re.sub(r"[^a-z0-9']+", " ", text)
        return text.split()

    def encode(self, caption: str, max_len: Optional[int] = None) -> List[int]:
        tokens = self._tokenise(caption)
        if max_len is not None:
            tokens = tokens[: max_len - 2]
        unk = self.stoi[self.UNK_TOKEN]
        ids = [self.stoi[self.SOS_TOKEN]]
        ids += [self.stoi.get(t, unk) for t in tokens]
        ids += [self.stoi[self.EOS_TOKEN]]
        return ids

    def decode(self, indices: List[int], skip_special: bool = True) -> str:
        specials = {self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN, self.UNK_TOKEN}
        tokens = []
        for idx in indices:
            tok = self.itos[idx] if idx < len(self.itos) else self.UNK_TOKEN
            if tok == self.EOS_TOKEN:
                break
            if skip_special and tok in specials:
                continue
            tokens.append(tok)
        return " ".join(tokens)

    @property
    def pad_idx(self) -> int: return self.stoi[self.PAD_TOKEN]
    @property
    def sos_idx(self) -> int: return self.stoi[self.SOS_TOKEN]
    @property
    def eos_idx(self) -> int: return self.stoi[self.EOS_TOKEN]
    def __len__(self) -> int: return len(self.itos)

    def save(self, path: str) -> None:
        with open(path, "wb") as f:
            pickle.dump({"itos": self.itos, "stoi": self.stoi, "min_freq": self.min_freq}, f)
        logger.info(f"Vocabulary saved to {path}")

    @classmethod
    def load(cls, path: str) -> "Vocabulary":
        with open(path, "rb") as f:
            data = pickle.load(f)
        vocab = cls(min_freq=data["min_freq"])
        vocab.itos = data["itos"]
        vocab.stoi = data["stoi"]
        return vocab


# Build from TRAIN captions only — no leakage from validation
train_all_captions = [cap for caps in train_captions.values() for cap in caps]
vocab = Vocabulary(min_freq=cfg.vocab_min_freq)
vocab.build(train_all_captions)
vocab.save(f"{cfg.output_dir}/vocab.pkl")

print(f"Vocabulary size: {len(vocab):,}")
print(f"PAD={vocab.pad_idx}, SOS={vocab.sos_idx}, EOS={vocab.eos_idx}")

## 4. Dataset & DataLoader

The `CaptionDataset` maps each (image, caption) pair into tensors. Because each image has five captions, I treat each pair as an independent training example — this gives us ~40,000 training samples from 8,000 images, which is meaningful data augmentation in itself.

For training I use fairly aggressive augmentation (random crops, colour jitter, horizontal flip) to improve generalisation. The validation transform is just a clean resize + normalise — no stochastic augmentation there, since we want a stable loss estimate.

In [ ]:
def get_transforms(split: str, image_size: int) -> transforms.Compose:
    mean = [0.485, 0.456, 0.406]  # ImageNet stats
    std  = [0.229, 0.224, 0.225]

    if split == "train":
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:  # val / test
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


class CaptionDataset(Dataset):
    """
    Returns (image_tensor, caption_tensor) pairs.
    Each image is paired with ONE randomly selected caption during training
    to act as implicit data augmentation.
    """

    def __init__(
        self,
        captions_map: Dict[str, List[str]],
        images_dir: str,
        vocab: Vocabulary,
        transform: transforms.Compose,
        max_caption_len: int,
        split: str = "train",
    ):
        self.images_dir = Path(images_dir)
        self.vocab = vocab
        self.transform = transform
        self.max_caption_len = max_caption_len
        self.split = split

        # Flatten: one entry per (image, caption) pair
        self.pairs: List[Tuple[str, str]] = [
            (img, cap)
            for img, caps in captions_map.items()
            for cap in caps
        ]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_name, caption = self.pairs[idx]
        img_path = self.images_dir / img_name

        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)

        ids = self.vocab.encode(caption, max_len=self.max_caption_len)
        cap_tensor = torch.tensor(ids, dtype=torch.long)

        return img_tensor, cap_tensor


def collate_fn(batch):
    """Pad caption tensors in a batch to the same length."""
    imgs, caps = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    caps = pad_sequence(caps, batch_first=True, padding_value=vocab.pad_idx)
    return imgs, caps


train_transform = get_transforms("train", cfg.image_size)
eval_transform  = get_transforms("val",   cfg.image_size)  # shared by val + test

train_dataset = CaptionDataset(train_captions, cfg.images_dir, vocab,
                               train_transform, cfg.max_caption_len, split="train")
val_dataset   = CaptionDataset(val_captions,   cfg.images_dir, vocab,
                               eval_transform,  cfg.max_caption_len, split="val")
test_dataset  = CaptionDataset(test_captions,  cfg.images_dir, vocab,
                               eval_transform,  cfg.max_caption_len, split="test")

# seed_worker + generator make DataLoader shuffling reproducible
train_loader = DataLoader(
    train_dataset, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, collate_fn=collate_fn,
    pin_memory=True, drop_last=True,
    worker_init_fn=seed_worker, generator=_g,
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, collate_fn=collate_fn,
    pin_memory=True, worker_init_fn=seed_worker,
)
# test_loader is not used during training — it's here so inference.ipynb
# can import this config if needed, but the split.json file is the primary
# mechanism for passing test image names across notebooks.
test_loader = DataLoader(
    test_dataset, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, collate_fn=collate_fn,
    pin_memory=True, worker_init_fn=seed_worker,
)

print(f"Train: {len(train_dataset):,} pairs  |  Val: {len(val_dataset):,}  |  Test: {len(test_dataset):,}")

## 4. Model Definition

### Architecture overview

I use a standard **encoder–decoder** architecture:

- **Encoder** — EfficientNet-B3 (pretrained on ImageNet). I remove the classification head and use the feature map from the last convolutional stage. This gives us a grid of visual features (~7×7 spatial positions, 384 channels for B3). I project these to `embed_dim` with a learned linear layer so the decoder sees consistent-sized features regardless of which encoder backbone we choose.

- **Decoder** — a stack of standard Transformer decoder layers. Each word embedding attends to the encoder's spatial features via cross-attention, and to previously generated words via masked self-attention. Positional encodings are sinusoidal (no extra parameters).

Why a Transformer decoder rather than an LSTM? Transformers handle long-range dependencies more gracefully and are much more parallelisable during training — teacher-forced training of the full sequence is one matrix multiply per layer rather than a sequential loop.

In [ ]:
class ImageEncoder(nn.Module):
    """
    EfficientNet-B3 encoder (via timm) with a learned linear projection.
    Outputs a sequence of (B, S, embed_dim) spatial feature vectors.
    timm is a hard dependency — install with: pip install timm
    """

    def __init__(self, model_name: str, embed_dim: int, pretrained: bool = True):
        super().__init__()
        self.embed_dim = embed_dim

        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, features_only=True
        )
        # Probe the feature channel count with a dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            self.feat_channels = self.backbone(dummy)[-1].shape[1]

        self.proj = nn.Sequential(
            nn.Linear(self.feat_channels, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (B, 3, H, W)
        Returns:
            features: (B, S, embed_dim)  where S = H' * W' spatial positions
        """
        x = self.backbone(images)[-1]                     # (B, C, H', W')
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H * W, C)  # (B, S, C)
        x = self.proj(x)                                  # (B, S, embed_dim)
        return x

    def freeze(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze(self):
        for p in self.backbone.parameters():
            p.requires_grad = True


class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""

    def __init__(self, embed_dim: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, embed_dim)"""
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class CaptionDecoder(nn.Module):
    """Transformer decoder that generates captions one token at a time."""

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_heads: int,
        num_layers: int,
        feedforward_dim: int,
        dropout: float,
        pad_idx: int,
        max_len: int = 512,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.pad_idx = pad_idx

        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(embed_dim, dropout=dropout, max_len=max_len)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,  # Pre-LN — more stable training
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embed_dim),
        )
        self.fc_out = nn.Linear(embed_dim, vocab_size)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.token_embed.weight, std=0.02)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            tgt:    (B, T)      — caption tokens (teacher-forced during training)
            memory: (B, S, D)   — encoder output
            tgt_key_padding_mask: (B, T) bool mask, True where token is PAD
        Returns:
            logits: (B, T, vocab_size)
        """
        T = tgt.size(1)
        # Causal mask — prevents attending to future tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)

        x = self.token_embed(tgt) * math.sqrt(self.embed_dim)
        x = self.pos_enc(x)

        out = self.transformer_decoder(
            tgt=x,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )
        logits = self.fc_out(out)  # (B, T, vocab_size)
        return logits


class ImageCaptioningModel(nn.Module):
    """Encoder–decoder image captioning model."""

    def __init__(self, encoder: ImageEncoder, decoder: CaptionDecoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(
        self,
        images: torch.Tensor,
        captions: torch.Tensor,
    ) -> torch.Tensor:
        """
        Teacher-forced forward pass.
        Args:
            images:   (B, 3, H, W)
            captions: (B, T)  — full token sequence including SOS and EOS
        Returns:
            logits: (B, T-1, vocab_size)  — predictions for tokens 1..T
        """
        memory = self.encoder(images)           # (B, S, D)
        tgt = captions[:, :-1]                  # Input: SOS .. second-to-last token
        pad_mask = (tgt == self.decoder.pad_idx)
        logits = self.decoder(tgt, memory, tgt_key_padding_mask=pad_mask)
        return logits

    @torch.no_grad()
    def generate(
        self,
        images: torch.Tensor,
        sos_idx: int,
        eos_idx: int,
        max_len: int = 50,
        beam_size: int = 1,
    ) -> List[List[int]]:
        """
        Greedy (beam_size=1) or beam-search generation.
        Returns a list of token-index lists, one per image in the batch.
        """
        self.eval()
        memory = self.encoder(images)  # (B, S, D)

        if beam_size == 1:
            return self._greedy_decode(memory, sos_idx, eos_idx, max_len)
        else:
            # Beam search — batched implementation (one image at a time)
            results = []
            for i in range(memory.size(0)):
                mem_i = memory[i: i + 1]  # (1, S, D)
                result = self._beam_search(mem_i, sos_idx, eos_idx, max_len, beam_size)
                results.append(result)
            return results

    def _greedy_decode(
        self, memory: torch.Tensor, sos_idx: int, eos_idx: int, max_len: int
    ) -> List[List[int]]:
        B = memory.size(0)
        device = memory.device
        tokens = torch.full((B, 1), sos_idx, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        results = [[] for _ in range(B)]

        for _ in range(max_len):
            logits = self.decoder(tokens, memory)  # (B, T, vocab)
            next_token = logits[:, -1, :].argmax(dim=-1)  # (B,)
            tokens = torch.cat([tokens, next_token.unsqueeze(1)], dim=1)

            for b in range(B):
                if not finished[b]:
                    if next_token[b].item() == eos_idx:
                        finished[b] = True
                    else:
                        results[b].append(next_token[b].item())

            if finished.all():
                break

        return results

    def _beam_search(
        self,
        memory: torch.Tensor,  # (1, S, D)
        sos_idx: int,
        eos_idx: int,
        max_len: int,
        beam_size: int,
    ) -> List[int]:
        device = memory.device
        # Each beam: (score, token_list)
        beams = [(0.0, [sos_idx])]
        completed = []

        for _ in range(max_len):
            candidates = []
            for score, seq in beams:
                if seq[-1] == eos_idx:
                    completed.append((score, seq))
                    continue
                inp = torch.tensor([seq], dtype=torch.long, device=device)
                mem_exp = memory.expand(1, -1, -1)
                logits = self.decoder(inp, mem_exp)[:, -1, :]  # (1, vocab)
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)  # (vocab,)
                topk = log_probs.topk(beam_size)
                for prob, idx in zip(topk.values, topk.indices):
                    candidates.append((score + prob.item(), seq + [idx.item()]))

            if not candidates:
                break
            candidates.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
            beams = candidates[:beam_size]

        completed += beams
        best = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
        # Strip SOS/EOS
        return [t for t in best[1] if t not in (sos_idx, eos_idx)]


# ── Instantiate model ─────────────────────────────────────────────────────────
encoder = ImageEncoder(
    model_name=cfg.encoder_name,
    embed_dim=cfg.embed_dim,
    pretrained=cfg.encoder_pretrained,
)
decoder = CaptionDecoder(
    vocab_size=len(vocab),
    embed_dim=cfg.embed_dim,
    num_heads=cfg.num_heads,
    num_layers=cfg.num_decoder_layers,
    feedforward_dim=cfg.feedforward_dim,
    dropout=cfg.dropout,
    pad_idx=vocab.pad_idx,
    max_len=cfg.max_caption_len + 10,
)
model = ImageCaptioningModel(encoder, decoder).to(cfg.device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training

A few design choices worth explaining:

- **Label smoothing (ε = 0.1)** — instead of maximising log-probability on the single ground-truth token, the model is trained to put (1 − ε) on the correct token and distribute ε uniformly across the rest. This prevents over-confident predictions and tends to produce better-calibrated outputs.

- **Encoder warmup** — for the first `warmup_epochs` epochs, the encoder backbone is frozen and only the projection layer and decoder are trained. Then the encoder is unfrozen with a lower learning rate. This avoids corrupting pretrained features at the start of training when the decoder is still random.

- **Cosine annealing with warm restarts** — the learning rate follows a cosine schedule, which consistently outperforms constant or step-decay schedules on vision tasks.

- **Gradient clipping** — clips the global L2 norm of gradients to 1.0. This is important for Transformers, which can experience gradient explosions during early training.

In [ ]:
# ── Loss & optimiser ──────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    ignore_index=vocab.pad_idx,
    label_smoothing=cfg.label_smoothing,
)

def make_optimizer(model: ImageCaptioningModel, cfg: Config) -> torch.optim.Optimizer:
    """Separate parameter groups so the encoder can use a lower LR."""
    encoder_params = list(model.encoder.backbone.parameters())
    other_params   = [p for p in model.parameters()
                      if not any(p is ep for ep in encoder_params)]
    return torch.optim.AdamW(
        [
            {"params": other_params,   "lr": cfg.learning_rate},
            {"params": encoder_params, "lr": cfg.learning_rate * cfg.encoder_lr_multiplier},
        ],
        weight_decay=cfg.weight_decay,
    )


optimizer = make_optimizer(model, cfg)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Freeze encoder during warmup
model.encoder.freeze()
logger.info("Encoder frozen for warmup phase.")


# ── Training & validation steps ───────────────────────────────────────────────
def train_epoch(
    model: ImageCaptioningModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    grad_clip: float,
) -> float:
    model.train()
    total_loss = 0.0
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    for imgs, caps in tqdm(loader, desc="  train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        caps = caps.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(imgs, caps)          # (B, T-1, vocab)
            targets = caps[:, 1:]               # (B, T-1)  — shift by one
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), targets.reshape(B * T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(
    model: ImageCaptioningModel,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> float:
    model.eval()
    total_loss = 0.0

    for imgs, caps in tqdm(loader, desc="  val  ", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        caps = caps.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(imgs, caps)
            targets = caps[:, 1:]
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), targets.reshape(B * T))

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
from dataclasses import asdict

# ── Main training loop ────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "lr": []}
best_val_loss = float("inf")
best_checkpoint_path = f"{cfg.output_dir}/best_model.pt"

logger.info(f"Starting training for {cfg.num_epochs} epochs on {cfg.device}")

for epoch in range(1, cfg.num_epochs + 1):
    t0 = time.time()

    # Unfreeze encoder after warmup
    if epoch == cfg.warmup_epochs + 1 and cfg.fine_tune_encoder:
        model.encoder.unfreeze()
        logger.info(f"Epoch {epoch}: encoder unfrozen — fine-tuning end-to-end.")

    train_loss = train_epoch(model, train_loader, optimizer, criterion,
                              cfg.device, cfg.grad_clip)
    val_loss   = val_epoch(model, val_loader, criterion, cfg.device)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    elapsed = time.time() - t0
    logger.info(
        f"Epoch {epoch:3d}/{cfg.num_epochs} | "
        f"train={train_loss:.4f} | val={val_loss:.4f} | "
        f"lr={current_lr:.2e} | {elapsed:.1f}s"
    )

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "cfg": asdict(cfg),   # plain dict — safe to load in any notebook
                "vocab_size": len(vocab),
            },
            best_checkpoint_path,
        )
        logger.info(f"  ✓ New best val loss: {val_loss:.4f} — checkpoint saved.")

    # Save latest checkpoint every 5 epochs (for resume)
    if epoch % 5 == 0:
        torch.save(
            {"epoch": epoch, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "val_loss": val_loss},
            f"{cfg.output_dir}/checkpoint_epoch{epoch:03d}.pt",
        )

logger.info(f"Training complete. Best val loss: {best_val_loss:.4f}")

## 6. Training Progress

Below I plot the loss curves and learning-rate schedule. A few things I look for:

- **Both curves decreasing** — the model is learning.
- **Val loss staying close to train loss** — no severe overfitting. If val diverges, I'd increase dropout or reduce the number of decoder layers.
- **No sudden spikes** — if I see them, it usually indicates the gradient clip is too loose or the LR is too high.

In [ ]:
# Save history to disk
with open(f"{cfg.output_dir}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss curves
ax1.plot(epochs_range, history["train_loss"], label="Train",
         color="royalblue", linewidth=2)
ax1.plot(epochs_range, history["val_loss"], label="Validation",
         color="tomato", linewidth=2, linestyle="--")
if cfg.warmup_epochs > 0:
    ax1.axvline(cfg.warmup_epochs, color="grey", linestyle=":",
                label=f"Encoder unfrozen (epoch {cfg.warmup_epochs})")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(alpha=0.3)

# LR schedule
ax2.plot(epochs_range, history["lr"], color="forestgreen", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Learning rate")
ax2.set_title("Learning Rate Schedule")
ax2.set_yscale("log")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.output_dir}/loss_curves.png", dpi=140)
plt.show()

print(f"Best validation loss: {best_val_loss:.4f} (perplexity ≈ {math.exp(best_val_loss):.1f})")

## Summary

At this point, all artefacts needed for inference are saved to `./artifacts/`:

| File | Contents |
|------|----------|
| `vocab.pkl` | Vocabulary built from **train captions only** |
| `split.json` | Sorted lists of `train_images`, `val_images`, `test_images` |
| `best_model.pt` | Weights + cfg dict at the best **val** loss |
| `checkpoint_epoch*.pt` | Periodic checkpoints for training resume |
| `training_history.json` | Loss + LR per epoch |
| `loss_curves.png` | Visual confirmation of learning progress |

The inference notebook (`inference.ipynb`) loads `vocab.pkl`, `best_model.pt`, and `split.json`, then runs the final demo and BLEU evaluation on **`test_images` only** — a set that was never touched during training or checkpoint selection.